# 🌤️ Weather Data Analysis
**Dataset:** 366 days of weather observations  
**Goal:** Explore temperature patterns, precipitation, wind, and predict rain

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

df = pd.read_csv('weather.csv')
df['Date'] = pd.date_range(start='2024-01-01', periods=len(df), freq='D')
df['Month'] = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.strftime('%b')
df['Season'] = df['Month'].map({
    12:'Summer', 1:'Summer', 2:'Summer',
    3:'Autumn', 4:'Autumn', 5:'Autumn',
    6:'Winter', 7:'Winter', 8:'Winter',
    9:'Spring', 10:'Spring', 11:'Spring'
})

print(f'Dataset shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head()

## 1. Temperature Trends Throughout the Year

In [ ]:
monthly = df.groupby('Month').agg(
    MinTemp=('MinTemp', 'mean'),
    MaxTemp=('MaxTemp', 'mean'),
    MeanTemp=('Temp9am', 'mean'),
    Rainfall=('Rainfall', 'sum')
).reset_index()
monthly['MonthName'] = pd.to_datetime(monthly['Month'], format='%m').dt.strftime('%b')

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=monthly['MonthName'], y=monthly['MaxTemp'],
    name='Max Temp', line=dict(color='#e74c3c', width=2.5)), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly['MonthName'], y=monthly['MinTemp'],
    name='Min Temp', line=dict(color='#3498db', width=2.5),
    fill='tonexty', fillcolor='rgba(52,152,219,0.1)'), secondary_y=False)
fig.add_trace(go.Bar(x=monthly['MonthName'], y=monthly['Rainfall'],
    name='Rainfall (mm)', marker_color='rgba(46,204,113,0.5)'), secondary_y=True)

fig.update_layout(title='Monthly Temperature Range & Rainfall',
    hovermode='x unified', template='plotly_white')
fig.update_yaxes(title_text='Temperature (°C)', secondary_y=False)
fig.update_yaxes(title_text='Rainfall (mm)', secondary_y=True)
fig.show()

## 2. Seasonal Statistics

In [ ]:
seasonal = df.groupby('Season').agg(
    AvgMaxTemp=('MaxTemp', 'mean'),
    AvgMinTemp=('MinTemp', 'mean'),
    TotalRainfall=('Rainfall', 'sum'),
    RainyDays=('RainToday', lambda x: (x == 'Yes').sum()),
    AvgHumidity=('Humidity3pm', 'mean'),
    AvgSunshine=('Sunshine', 'mean')
).round(1)

print('=== Seasonal Summary ===')
print(seasonal.to_string())

season_order = ['Summer', 'Autumn', 'Winter', 'Spring']
colors = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

s = seasonal.reindex(season_order)
axes[0].bar(season_order, s['AvgMaxTemp'], color=colors)
axes[0].set_title('Avg Max Temperature by Season')
axes[0].set_ylabel('°C')

axes[1].bar(season_order, s['TotalRainfall'], color=colors)
axes[1].set_title('Total Rainfall by Season')
axes[1].set_ylabel('mm')

axes[2].bar(season_order, s['RainyDays'], color=colors)
axes[2].set_title('Rainy Days by Season')
axes[2].set_ylabel('Days')

plt.tight_layout()
plt.savefig('seasonal_stats.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Correlation Heatmap

In [ ]:
numeric_cols = ['MinTemp','MaxTemp','Rainfall','Sunshine','WindGustSpeed',
                'Humidity9am','Humidity3pm','Pressure9am','Temp9am','Temp3pm','RISK_MM']
corr = df[numeric_cols].corr()

plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Wind Analysis — Rose Chart

In [ ]:
wind_rain = df.groupby('WindGustDir').agg(
    TotalRain=('Rainfall', 'sum'),
    AvgSpeed=('WindGustSpeed', 'mean'),
    Count=('WindGustDir', 'count')
).reset_index().dropna()

fig = px.bar_polar(
    wind_rain, r='TotalRain', theta='WindGustDir',
    color='AvgSpeed', color_continuous_scale='Blues',
    title='Rainfall by Wind Gust Direction (color = avg speed km/h)',
    template='plotly_white'
)
fig.update_layout(polar=dict(radialaxis=dict(showticklabels=True)))
fig.show()

print('\nTop 5 directions by total rainfall:')
print(wind_rain.nlargest(5, 'TotalRain')[['WindGustDir','TotalRain','AvgSpeed']].to_string(index=False))

## 5. Humidity vs Temperature vs Rain

In [ ]:
fig = px.scatter(
    df, x='Temp3pm', y='Humidity3pm',
    color='RainTomorrow', size='Rainfall',
    hover_data=['WindGustSpeed', 'Pressure3pm', 'Season'],
    title='Temperature vs Humidity at 3pm (size = rainfall, color = rain tomorrow)',
    color_discrete_map={'Yes': '#e74c3c', 'No': '#3498db'},
    template='plotly_white', opacity=0.7
)
fig.show()

## 6. Rain Prediction — Random Forest Classifier

In [ ]:
features = ['MinTemp','MaxTemp','Rainfall','Sunshine','WindGustSpeed',
            'Humidity9am','Humidity3pm','Pressure9am','Pressure3pm',
            'Temp9am','Temp3pm','Cloud9am','Cloud3pm']

df_ml = df[features + ['RainTomorrow']].dropna().copy()
df_ml['RainTomorrow'] = (df_ml['RainTomorrow'] == 'Yes').astype(int)

X = df_ml[features]
y = df_ml['RainTomorrow']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('=== Rain Tomorrow Prediction ===')
print(classification_report(y_test, y_pred, target_names=['No Rain', 'Rain']))

importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

importances.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Feature Importance')
axes[0].set_xlabel('Importance')

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['No Rain','Rain'], yticklabels=['No Rain','Rain'])
axes[1].set_title('Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('rain_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Insights

In [ ]:
rain_pct = (df['RainTomorrow'] == 'Yes').mean() * 100
hottest_month = monthly.loc[monthly['MaxTemp'].idxmax(), 'MonthName']
coldest_month = monthly.loc[monthly['MinTemp'].idxmin(), 'MonthName']
wettest_month = monthly.loc[monthly['Rainfall'].idxmax(), 'MonthName']
top_wind = wind_rain.nlargest(1, 'TotalRain')['WindGustDir'].values[0]

print('=' * 45)
print('           KEY INSIGHTS')
print('=' * 45)
print(f'  Rain probability:    {rain_pct:.1f}% of days')
print(f'  Hottest month:       {hottest_month}')
print(f'  Coldest month:       {coldest_month}')
print(f'  Wettest month:       {wettest_month}')
print(f'  Rainiest wind dir:   {top_wind}')
print(f'  ML model accuracy:   ~{(y_pred == y_test).mean()*100:.0f}%')
print('=' * 45)